This code implements TD3 for CF-mMIMO. For this task, you have been provided with fully functional code. 
Your task is to understand this code and identify:
1. From your understanding of the project, what should be the state, action, reward and next state. Explain your answer
2. From this code, how are the above variables obtained from the environment
   You can follow a tutorial for writing your won envirenment here: https://docs.pytorch.org/rl/main/tutorials/pendulum.html
4. Explain how this algorithm works
5. Observe the plotted training curve. Can you explain this curve?
6. Run this code for 100 realizations and plot the CDF curves for the sum SE and per user SE. Compare your results with those for the LSF gain algorithm. Note: ideally you would run inference on your trained NNs but for this assignment; simply get an average of the last 10 training values (after the algorithm converges)

In [ ]:
from scipy.io import savemat
import copy
import numpy as np
from Environment_SINR_random_binary_speedup import CfMIMOEnv
from Wolpertinger_TD3_guided_binary_nonbrs_masked_speedup import WolpertingerAgent
import matplotlib.pyplot as plt
import torch
import random
from tensordict.tensordict import TensorDict, TensorDictBase
from torchrl.data import BoundedTensorSpec, CompositeSpec, UnboundedContinuousTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
    DoubleToFloat,
)
from torchrl.envs.transforms.transforms import _apply_to_composite
import time
from evaluator_binary import Evaluator
from util import*

Define training loop

In [ ]:
def train(num_iterations,agent,env,evaluate,validate_steps,output,max_episode_length=None,debug=False):
    agent.is_training=True
    step=episode=episode_steps=0
    episode_reward=0.
    observation=None
    episode_rewardlist=[]
    track_reward=[]
    track_episode_steps=[]
    episode_rewards=[]
    betarate=[]
    step_time=[]
    update_steps=0
    
    while step<num_iterations:
        
        starttime=time.time()
        if observation is None:
            terminated=False
            truncated=False
            td_obs=env.reset()
            s_t=[np.array(td_obs["normalized_gainovernoise"].view(args.UEnumber*args.APnumber))]
            observation=copy.deepcopy(s_t) 
            agent.reset(observation) 
            
        ## pick action
        if step<=args.warmup:
            ## pick a random action
            action=agent.random_action(args.UEnumber,td_obs["gainovernoise"],td_obs["APMask"],td_obs["pilotIndex"],td_obs["CandidateAPs"])
            
        else:
           action,action_val,proto_action=agent.select_action(observation,td_obs["gainovernoise"],td_obs["APMask"],td_obs["pilotIndex"],td_obs["CandidateAPs"])
           
        #### carry out a step in the environment
        
        td_obs["action"]=action.squeeze()
        td_obs=env.step(td_obs)
        ### get current state
       
        s_t=[np.array(td_obs["normalized_gainovernoise"].view(args.UEnumber*args.APnumber))]
        observation=s_t
        ## get new state after action
        
        s_t2=[np.array(td_obs["next","normalized_gainovernoise"].view(args.UEnumber*args.APnumber))]
        
        observation2,reward,done=s_t2,np.array(td_obs["next","reward"]),np.array(td_obs["next","done"])
        observation2=copy.deepcopy(observation2)
       
        ## update terminated 
        terminated=done #episode reached terminal state
        
        if max_episode_length and episode_steps >= max_episode_length-1:
            done=True
            #update truncated
            truncated=True
    
        #### agent observes values from step and updates policy
        
        # warmup for real buffer
        if step<args.warmup:
            agent.replay_buffer.add(s_t[0], action, reward, s_t2[0], done, terminated)
       
        ##########################
        if step > args.warmup:
            update_steps+=1
           
            agent.update_policy(td_obs["gainovernoise"],step,td_obs["APMask"],td_obs["pilotIndex"],td_obs["CandidateAPs"]) 
                 
        ### [optional evaluate]
        if evaluate is not None and validate_steps > 0 and step % validate_steps==0:
            policy = lambda x: agent.select_action(x,td_obs["gainovernoise"],td_obs["APMask"],td_obs["pilotIndex"],td_obs["CandidateAPs"],SA_mode=args.SA_mode)
            validate_reward=evaluate(env,policy,args,td_obs["APMask"],td_obs["CandidateAPs"],debug=False,visualize=False)
            #eval_step_reward.append(eval_reward)
            if debug: prYellow('[Evaluate] Step_{:07d}: mean_reward:{}'.format(step,validate_reward))
            
        # optional save intermediate model
        if step % int(num_iterations/3)==0:
            agent.save_model(output)
        
        # update 
        step+=1 
       
        episode_steps+=1 
       
        episode_reward+=reward 
        episode_rewards.append(reward)
        track_reward.append(reward)
        
        ### update input state because environment is stateless
        td_obs["ServingAPs"]=copy.deepcopy(td_obs["next","ServingAPs"] )
        td_obs["SINR"]=copy.deepcopy(td_obs["next","SINR"] )
        td_obs["APsperUE"]=copy.deepcopy(td_obs["next","APsperUE"] )
        td_obs["UEsperAP"]=copy.deepcopy(td_obs["next","UEsperAP"] )
        td_obs["normalized_gainovernoise"]=copy.deepcopy(td_obs["next","normalized_gainovernoise"])
        observation=copy.deepcopy(observation2)
        
        if done:
            # episode has ended
            
            ##get beta  actions
            betaa_orig=BetaAction(args.APnumber, args.UEnumber, np.array(td_obs["gainovernoise"]))
            
            
            
            brate1,_=compute_rates(args.APnumber,args.UEnumber, betaa_orig, \
                                          td_obs["gainovernoise"], td_obs["params","Pd"], td_obs["params","tau_p"], td_obs["params","tau_c"], td_obs["B"], \
                                              td_obs["R"], td_obs["pilotIndex"]) 
            
           
            brate=np.sum(brate1.numpy())
                
            if debug: prGreen('#{}: episode_reward:{} steps:{}'.format(episode,episode_reward,step))
           
            episode_rewardlist.append(episode_reward)
            
            
            track_episode_steps.append(episode_steps)
            #start new trajectory
            observation=None
            #######delete
            
            betarate.append(episode_steps*brate) 
            
           
            episode_steps=0
            episode_reward=0
            
            episode_rewards=[]
            
            episode+=1
    
        endtime=time.time()
        step_time.append(endtime-starttime)
        
    
    
    return(track_reward,episode_rewardlist,track_episode_steps,\
           betarate\
               )
                    

Define test loop

In [ ]:
def test(num_episodes,agent,env,evaluate,model_path,visualize=False,debug=False):
    agent.load_weights(model_path)
    agent.is_training=False
    agent.eval()
    ###need gain over noise so reset
    td=env.reset()
    policy=lambda x: agent.select_action(x,td["gainovernoise"],td["APMask"],td["pilotIndex"],td["CandidateAPs"],SA_mode=args.SA_mode)
    
    
    for i in range(1): ### episodes already  passed to validate
   
        validate_reward=evaluate(env,policy,args,td["APMask"],td["CandidateAPs"],debug=debug,visualize=visualize,save=True)
        if debug: prYellow('[Evaluate] #{}: mean_reward:{}'.format(i,validate_reward))

Compute serving APs based on LSF gain

In [ ]:
def BetaAction(M,K,gainovernoise): 
    ServingAPsBeta=np.zeros((M,K),dtype=int) 
   
    for k in range(K): 
        
        gain=gainovernoise[:,k]
        sorted_gain=np.sort(gain)[::-1]## sort in descending order
        current_gain=np.zeros(M)
        for m in range(M): 
            current_gain[m]=np.sum(sorted_gain[:m+1])/np.sum(sorted_gain)
        #ipdb.set_trace()
        stopindex=np.where(current_gain>=0.95)[0][0]
        #ipdb.set_trace()
        for id in range(stopindex+1):
            AP=np.where(gain==sorted_gain[id])[0]
            ServingAPsBeta[AP,k]=1
   
    return(ServingAPsBeta)

Function to compute rates

In [ ]:
def compute_rates(M,K,APs,gainovernoise,Pd,tau_p,tau_c,B,R,pilotIndex):
    
    Power=np.zeros((M,K),dtype=float)
    for m in range(M):
        servedUEs=np.where(APs[m,:]==1)[0]
        #ipdb.set_trace()
        denominator=np.sum(np.sqrt(gainovernoise[m,servedUEs]).numpy())
        for k in servedUEs:
            Power[m,k]=Pd*np.sqrt(gainovernoise[m,k])/denominator  
#compute SEs
    signal_MR=np.zeros(K,dtype='float')
    interf_MR=np.zeros(K,dtype='float')
    cont_MR=np.zeros((K,K),dtype='float')
    pre_logfactor=1-tau_p/tau_c
    for m in range(M):
        servedUEs=np.where(APs[m,:]==1)[0]
        for k in servedUEs:
            signal_MR[k]=signal_MR[k]+np.sqrt(Power[m,k]*np.real(np.trace(B[:,:,m,k])))
            for i in range(K):
                interf_MR[i]=interf_MR[i]+Power[m,k]*\
                    np.real(np.trace(np.matmul(B[:,:,m,k],R[:,:,m,i])))/np.real(np.trace(B[:,:,m,k]))
                if pilotIndex[k]==pilotIndex[i]:
                    cont_MR[i,k]=cont_MR[i,k]+np.sqrt(Power[m,k])*\
                        np.real(np.trace(np.matmul((np.matmul(B[:,:,m,k],np.linalg.inv(R[:,:,m,k]))),R[:,:,m,i])))\
                            /np.sqrt(np.real(np.trace(B[:,:,m,k])))
    SINR=(np.abs(signal_MR)**2)/(interf_MR+np.sum(np.abs(cont_MR)**2,axis=1)-np.abs(signal_MR)**2+1)
    SE_MR=pre_logfactor*np.real(np.log2(1+SINR)) 
    return(SE_MR,SINR)

Define parameters

In [ ]:
class Arguments(object):
    def __init__(self):
        self.mode = 'train'
        self.UEnumber=5 ## number of UES
        self.APnumber=12 ### number of APs
        self.tau_p=2 #cnumber of pilots 
        self.env = "CFmimo" #envirenment 
        self.hidden1 = 256 #layer 1 hidden layers
        self.hidden2 = 256 # layer 2 hidden layers
        self.rate = 0.001 # learning rate for critic
        self.prate =0.001 #learning rate for actor
        self.warmup = 10000 ## populate replay buffer with initial data - speeds up training 
        self.discount =0.99 # discount factor
        self.bsize =256 #minibatch size
        self.rmsize = 100000 #replay buffer size
        self.window_length = 1 
        self.tau = 0.005 # soft update parameter
        self.validate_episodes = 1 
        self.max_episode_length = 100 #episode length
        self.validate_steps = 10000
        self.output = 'output'
        self.debug='debug'
        self.init_wt = 0.003 
        self.train_iter=20000  # total number of iterations
        self.epsilon=50000
        self.seed=20
        self.max_actions=1e6
        self.resume='default'
        self.k_ratio = 1e-6
        self.noise='gaussian' #or OU
        self.sigma=0.1 # sigma for gaussian noise for exploration 
        self.SA_mode=False #for greedy or True for SA
        self.noise_clip=0.5
        self.update_freq=2
        self.sigma_target=0.2 #sigma noise for target 
        self.adaptive_alpha=True
        self.alpha=0.12 #entropy coefficient
        self.dvc=torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu') #or cuda
        self.candidate_no=0.00 #percentage of APs in cadidate set for nbrs

Train Neural network 

In [ ]:

seeds=[0] # set seed for reproducibility 
#seeds=[100]
#seeds=np.arange(100).tolist()
### when you finish making the code work well, run this or 100 seeds 0 to 99
for trial in range(len(seeds)):
    
    args = Arguments()
   
    args.seed=seeds[trial]
    
    args.output=get_output_folder(args.output,args.env,args.seed)
    if args.resume=='default':
        args.resume='output/{}-run0'.format(args.env)


   
#### make env
## call environment
    env=CfMIMOEnv(args.UEnumber,args.APnumber,args.tau_p,args.candidate_no,seed=args.seed)
    env=TransformedEnv(
        env,
        
    )
    #normalize beta values with max-min to scale between 0 and 1
    #see: https://pytorch.org/tutorials/advanced/pendulum.html
    class NormalizeBetaTransform(Transform):
        def _apply_transform(self, obs: torch.Tensor) -> None:
           
            return (obs-obs.mean())/(obs.std())
           
    
        # The transform must also modify the data at reset time
        def _reset(
            self, tensordict: TensorDictBase, tensordict_reset: TensorDictBase
        ) -> TensorDictBase:
            return self._call(tensordict_reset)
    
        # _apply_to_composite will execute the observation spec transform across all
        # in_keys/out_keys pairs and write the result in the observation_spec which
        # is of type ``Composite``
        @_apply_to_composite
        def transform_observation_spec(self, observation_spec):
            return BoundedTensorSpec(
                low=-1,
                high=1,
                shape=observation_spec.shape,
                dtype=observation_spec.dtype,
                device=observation_spec.device,
            )
    
    t_normbeta = NormalizeBetaTransform(in_keys=["gainovernoise"], out_keys=["normalized_gainovernoise"])
    env.append_transform(t_normbeta)
    env.append_transform(DoubleToFloat()) ## models work with float 32
    
    args.low=env.action_spec.space.low[0].item()
    args.high=env.action_spec.space.high[0].item()
    if args.seed>0:
        np.random.seed(args.seed)
        random.seed(args.seed)
        #env.seed(args.seed)
        torch.manual_seed(args.seed)
    
    td_init=env.reset()
    number_of_states=args.UEnumber*args.APnumber
    number_of_actions=args.UEnumber*args.APnumber
    
    ##call wolpertinger agent
    agent=WolpertingerAgent(number_of_states, number_of_actions, args)
    ####1st pause evaluate
    evaluate=None
    #evaluate=Evaluator(args.validate_episodes,args.validate_steps,args.UEnumber,args.output,SA_mode=args.SA_mode,
                        #max_episode_length=args.max_episode_length)
    
    if args.mode=='train':
        
       
        track_reward,episode_reward,eps_steps,betarate=\
                train(args.train_iter,agent,env,evaluate,args.validate_steps,args.output,\
              max_episode_length=args.max_episode_length,debug=args.debug)
        savemat('{}/train_results.mat'.format(args.output),{'Step_reward':track_reward,'Episode_reward':episode_reward,'Episode_steps':eps_steps,\
            'Beta_rate':betarate,\
            })   ### save your data in matlab files and then you can process them altogether at the end         
       
       
       #### visualize data. you should comment this code when you are running multiple iterations
        
        plt.figure(1)
        plt.plot(track_reward)
        
        betarate_steps=np.tile(betarate[0]/eps_steps[0],args.train_iter)
        
        plt.plot(betarate_steps)
        
        plt.legend(["ml","beta"])
        plt.ylabel('step_reward')
        plt.xlabel('step')
        plt.show()
       
        #######################
        
        
    elif args.mode=='test':
        test(args.validate_episodes,agent,env,evaluate,args.resume,
             visualize=True,debug=args.debug)
    else:
        raise RunTimeError('undefined mode{}'.format(args.mode))





    